# Homework Starter — Stage 13: Productization

**This homework is self-contained.** It does not use your project data or your project
model — the cells below generate everything they need. Work through it in order.

You are building four things: a saved model, a Flask app that loads it at startup and
serves two routes, proof from this notebook that both routes work, and a README that
tells someone else how to call them.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install scikit-learn
# !pip install joblib
# !pip install flask
# !pip install requests

## 1. Generate data and train a model

Nothing to fill in here — run it. Note `os.makedirs` **before** `joblib.dump`: without it
the save fails with `FileNotFoundError`, because `model/` does not exist yet.

In [2]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# the dataset for this homework - generated, not loaded
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

model = LinearRegression()
model.fit(X, y)

os.makedirs('model', exist_ok=True)          # BEFORE the dump, not after
joblib.dump(model, 'model/model.pkl')

# prove the file on disk is usable: load it back and predict with the loaded copy
reloaded = joblib.load('model/model.pkl')
print('saved to model/model.pkl')
print('prediction from the reloaded model:', reloaded.predict([[0.1, 0.2]])[0])

saved to model/model.pkl
prediction from the reloaded model: 23.58961171297328


## 2. Write `app.py`

Fill in the three TODOs, then run the cell — it writes `app.py` to disk.

**The model load stays where it is**, at the top of the file. It runs once when the app
starts. Do not move it inside a route: a route that loads the model on every request
re-reads the file from disk for every single caller.

In [3]:
app_code = '''
from flask import Flask, request, jsonify
import joblib

# Load the model ONCE when the app starts.
# Do not move this inside either route.
model = joblib.load('model/model.pkl')

app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}

    features = data.get('features')

    # Check that features exists and has exactly 2 values
    if (
        features is None
        or not isinstance(features, list)
        or len(features) != 2
    ):
        return jsonify({
            'error': 'features must be a list containing exactly 2 values'
        }), 400

    # Make sure both values are numeric
    try:
        features = [
            float(features[0]),
            float(features[1])
        ]
    except (TypeError, ValueError):
        return jsonify({
            'error': 'features must contain numeric values'
        }), 400

    prediction = model.predict(
        [features]
    )[0]

    return jsonify({
        'prediction': float(prediction)
    })


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):

    try:
        f1 = float(f1)
        f2 = float(f2)

    except ValueError:
        return jsonify({
            'error': 'f1 and f2 must be numeric'
        }), 400

    prediction = model.predict(
        [[f1, f2]]
    )[0]

    return jsonify({
        'prediction': float(prediction)
    })


if __name__ == '__main__':
    app.run(port=5000)
'''

with open(
    'app.py',
    'w'
) as f:
    f.write(app_code)

print('wrote app.py')

wrote app.py


## 3. Launch the server

This opens a **separate terminal window** and starts Flask there. Leave it running.
Every time you change `app.py`, close that window and run this cell again.

In [6]:
os.system("start cmd /k python app.py")
print('Flask launched in an external terminal window. Give it a few seconds to start.')

Flask launched in an external terminal window. Give it a few seconds to start.


## 4. Call your own API

Three calls: the POST route, the GET route, and one deliberately bad call. **Leave this
output visible in the notebook you submit — it is your testing evidence.**

In [7]:
import requests

BASE = 'http://127.0.0.1:5000'

try:
    r1 = requests.post(BASE + '/predict', json={'features': [0.1, 0.2]}, timeout=5)
    print('POST /predict          ', r1.status_code, r1.text.strip())

    r2 = requests.get(BASE + '/predict/0.1/0.2', timeout=5)
    print('GET  /predict/0.1/0.2  ', r2.status_code, r2.text.strip())

    # deliberately bad: not a number. This must be a 400 and a JSON error,
    # not a traceback in the server window.
    r3 = requests.get(BASE + '/predict/abc/0.2', timeout=5)
    print('GET  /predict/abc/0.2  ', r3.status_code, r3.text.strip())
except requests.exceptions.ConnectionError:
    print('No server on port 5000. Run the launch cell above, wait a few seconds,')
    print('then run this cell again.')

POST /predict           200 {"prediction":23.58961171297328}
GET  /predict/0.1/0.2   200 {"prediction":23.58961171297328}
GET  /predict/abc/0.2   400 {"error":"f1 and f2 must be numeric"}


In [8]:
r_bad_post = requests.post(
    BASE + '/predict',
    json={
        'features': [0.1]
    },
    timeout=5
)

print(
    'Bad POST status:',
    r_bad_post.status_code
)

print(
    'Bad POST response:',
    r_bad_post.json()
)

Bad POST status: 400
Bad POST response: {'error': 'features must be a list containing exactly 2 values'}


In [9]:
r_missing = requests.post(
    BASE + '/predict',
    json={},
    timeout=5
)

print(
    r_missing.status_code
)

print(
    r_missing.json()
)

400
{'error': 'features must be a list containing exactly 2 values'}


## 5. Write `README.md`

Run the cell to get a template, then **edit the file** — replace every `TODO` with the
real thing, and paste in the responses you actually got above.

In [10]:
print(
    r1.json()
)

print(
    r2.json()
)

{'prediction': 23.58961171297328}
{'prediction': 23.58961171297328}


In [14]:
from pathlib import Path

post_response = r1.text.strip()
get_response = r2.text.strip()
bad_response = r3.text.strip()

readme_lines = [
    "# Stage 13 Homework - Prediction API",
    "",
    "This project serves a linear regression model through a Flask API.",
    "The model takes two numeric features as input and returns a continuous numerical prediction.",
    "",
    "## Running the API",
    "",
    "From the `homework/homework13` directory, run:",
    "",
    "```bash",
    "python app.py",
    "```",
    "",
    "The server runs at `http://127.0.0.1:5000`.",
    "",
    "The trained model is loaded once from `model/model.pkl` when the Flask application starts.",
    "",
    "## POST /predict",
    "",
    "The POST route accepts a JSON body containing exactly two numeric features.",
    "",
    "Example request:",
    "",
    "```bash",
    'curl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" -d "{\\"features\\": [0.1, 0.2]}"',
    "```",
    "",
    "Example response:",
    "",
    "```json",
    post_response,
    "```",
    "",
    "## GET /predict/<f1>/<f2>",
    "",
    "The GET route accepts the two numeric features directly in the URL.",
    "",
    "Example request:",
    "",
    "```bash",
    "curl http://127.0.0.1:5000/predict/0.1/0.2",
    "```",
    "",
    "Example response:",
    "",
    "```json",
    get_response,
    "```",
    "",
    "## Bad Input",
    "",
    "Invalid input returns HTTP status code `400` and a JSON error message instead of a traceback.",
    "",
    "Example invalid request:",
    "",
    "```bash",
    "curl http://127.0.0.1:5000/predict/abc/0.2",
    "```",
    "",
    "Example response:",
    "",
    "```json",
    bad_response,
    "```",
    "",
    "The POST route also returns HTTP `400` when:",
    "",
    "- the `features` key is missing;",
    "- the `features` list does not contain exactly two values;",
    "- either feature is not numeric.",
]

readme = "\n".join(readme_lines)

Path("README.md").write_text(
    readme,
    encoding="utf-8"
)

print("README.md created successfully.")
print("Saved to:", Path("README.md").resolve())

README.md created successfully.
Saved to: E:\研究生\bootcamp\bootcamp_aixuan_liu\homework\homework13\README.md


### Save Notebook
Remember to save as `homework13_productization_submission.ipynb`.